In [7]:
import numpy as np
import lmdb
import pickle
import torch

result_npz_path = '/home/liud/Documents/code/safe-gemnet-oc/example/2026-10-14-23-10-56/is2rfse_predictions_0.npz'
mdb_path = '/home/liud/Documents/code/safe-gemnet-oc/dataset/test'

# 读取 npz 文件中的数据
data = np.load(result_npz_path)

sid_pre = torch.tensor(data['ids'].astype(np.int32))
sorted_sid_pre, indices_pre = torch.sort(sid_pre)
energy_pre = torch.tensor(data['energy'])[indices_pre]

energy_relaxed = []
sid_relaxed = []
env = lmdb.open(mdb_path, readonly=True)
with env.begin(write=False) as txn:
    cursor = txn.cursor()
    for key, value in cursor:
        info = pickle.loads(value)
        energy_relaxed.append(info.y)
        sid_relaxed.append(info.sid)
        
sid_relaxed = torch.tensor(sid_relaxed)
sorted_energy_relaxed, indices_relaxed = torch.sort(sid_relaxed)
energy_relaxed = torch.tensor(energy_relaxed)[indices_relaxed]
energy_mae = torch.mean(torch.abs(energy_relaxed - energy_pre))
print('energy_mae:', energy_mae)

e_thresh = 0.02
error_energy = torch.abs(energy_relaxed - energy_pre)
success = (error_energy < e_thresh).sum().item()
total = energy_relaxed.size(0)
success_rate = success / total
print('success rate:', success_rate)
e_thresh2 = 0.05
success2 = (error_energy < e_thresh2).sum().item()
success_rate2 = success2 / total
# print('success2:', success_rate2)

e_thresh3 = 0.1
success3 = (error_energy < e_thresh3).sum().item()
success_rate3 = success3 / total
# print('success3:', success_rate3)

energy_mae: tensor(0.0582, dtype=torch.float64)
success rate: 0.37886178861788616
